# OSHA ↔ CMS Nursing-Facility Crosswalk

Links establishments in OSHA's Injury Tracking Application (ITA) to nursing homes in CMS's Provider Information file, so that workplace-injury data and CMS quality data can be analyzed together.

OSHA data carries no CMS Certification Number (CCN). CMS data carries no OSHA establishment ID or employer identification number. The only shared information is name and address. This notebook matches on those, grades every match, and publishes the result as a table of identifiers.

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC), New York
**Source data** US Government work, 17 U.S.C. §105

---
## 1. Environment

In [1]:
!pip install -q duckdb pandas requests rapidfuzz scipy
import os, re, io, requests, duckdb, pandas as pd, numpy as np
from rapidfuzz import fuzz
from scipy import stats
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
con = duckdb.connect(':memory:')

def fetch(url, path):
    if os.path.exists(path): return True
    try:
        with requests.get(url, headers=HDRS, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path, 'wb') as f:
                for ch in r.iter_content(1 << 20): f.write(ch)
        return True
    except Exception as e:
        print('  not available:', url.split('/')[-1], '-', type(e).__name__); return False

---
## 2. OSHA establishments

The **Form 300A summary file** lists every establishment that submitted, including those with no recordable injuries. It is preferred because it allows injury rates. If it cannot be downloaded, the notebook falls back to the establishments that appear in the **case detail file**, which includes only establishments with at least one case.

File names change with each OSHA release. Update the URLs from osha.gov/Establishment-Specific-Injury-and-Illness-Data if both fail.

In [12]:
SUMMARY_URLS = ['https://www.osha.gov/sites/default/files/ITA_300A_Summary_Data_2025_through_03-15-2026_v2.csv']
CASES_URL = 'https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_3-15-2026.csv'

SOURCE = None
for u in SUMMARY_URLS:
    if fetch(u, 'data/osha_300a.csv'): SOURCE = '300A'; break
if SOURCE is None and fetch(CASES_URL, 'data/osha_cases.csv'): SOURCE = 'case_detail'
print('OSHA source:', SOURCE)

path = 'data/osha_300a.csv' if SOURCE == '300A' else 'data/osha_cases.csv'
con.execute(f"CREATE OR REPLACE TABLE osha_raw AS SELECT * FROM read_csv_auto('{path}', all_varchar=true, ignore_errors=true, sample_size=-1)")
ocols = {c.lower(): c for c in [r[0] for r in con.execute('DESCRIBE osha_raw').fetchall()]}
oc = lambda *n: next(ocols[x.lower()] for x in n if x.lower() in ocols)

agg = ''
if SOURCE == '300A':
    agg = f''', MAX(TRY_CAST("{oc('total_dafw_cases')}" AS INT)) AS dafw_cases,
               MAX(TRY_CAST("{oc('total_djtr_cases')}" AS INT)) AS djtr_cases'''
else:
    agg = ', COUNT(*) AS cases_reported'
osha = con.execute(f'''
SELECT "{oc('establishment_id')}" AS establishment_id,
       ANY_VALUE("{oc('establishment_name')}") AS establishment_name,
       ANY_VALUE("{oc('street_address')}") AS street_address,
       ANY_VALUE("{oc('city')}") AS city, ANY_VALUE("{oc('state')}") AS state,
       ANY_VALUE("{oc('zip_code','zip')}") AS zip_code,
       ANY_VALUE("{oc('naics_code')}") AS naics_code,
       MAX(TRY_CAST("{oc('annual_average_employees')}" AS INT)) AS employees,
       MAX(TRY_CAST("{oc('total_hours_worked')}" AS BIGINT)) AS hours {agg}
FROM osha_raw WHERE "{oc('naics_code')}" LIKE '6231%'
GROUP BY 1''').df()
print(f'{len(osha):,} OSHA nursing-facility establishments (NAICS 6231)')

OSHA source: 300A


8,975 OSHA nursing-facility establishments (NAICS 6231)


---
## 3. CMS nursing homes

CMS file URLs change monthly; the catalog at `data.cms.gov/provider-data/data.json` does not. The notebook finds the current *Provider Information* file for nursing homes from the catalog.

In [13]:
catalog = requests.get('https://data.cms.gov/provider-data/data.json', timeout=60).json()['dataset']
def cms_csv(title, theme_word='nursing'):
    hits = [d for d in catalog if d['title'].strip().lower() == title.lower()
            and theme_word in ' '.join(d.get('theme', [])).lower() + d.get('description', '').lower()]
    d = hits[0]
    url = next(x['downloadURL'] for x in d['distribution'] if x.get('downloadURL', '').endswith('.csv'))
    print(f"{d['title']} | modified {d.get('modified')}"); return url, d.get('modified')

prov_url, prov_date = cms_csv('Provider Information')
cms = pd.read_csv(prov_url, dtype=str)
ccol = {c.lower(): c for c in cms.columns}
cc = lambda *n: next(ccol[x.lower()] for x in n if x.lower() in ccol)
cms = cms.rename(columns={cc('CMS Certification Number (CCN)', 'Federal Provider Number'): 'ccn',
                          cc('Provider Name'): 'provider_name', cc('Provider Address'): 'provider_address',
                          cc('City/Town', 'Provider City'): 'provider_city', cc('State', 'Provider State'): 'provider_state',
                          cc('ZIP Code', 'Provider Zip Code'): 'provider_zip',
                          cc('Number of Certified Beds'): 'certified_beds'})
cms['certified_beds'] = pd.to_numeric(cms.certified_beds, errors='coerce')
print(f'{len(cms):,} CMS-certified nursing homes')

Provider Information | modified 2026-08-01
14,690 CMS-certified nursing homes


---
## 4. Normalization

Names and addresses are written differently in the two sources ("Nursing & Rehabilitation Center LLC" vs "NURSING AND REHAB CTR"; "Street" vs "ST"). Before comparing:

- **Addresses**: lower-case, strip punctuation and suite numbers, standardize street suffixes and directions, extract the house number.
- **Names**: lower-case, remove legal suffixes and generic words (nursing, rehabilitation, center, healthcare, and similar) that appear in most facility names and carry no identifying information.
- **ZIP**: first five digits.

In [14]:
import re, numpy as np, pandas as pd
from rapidfuzz import fuzz

SUFFIX = {'street':'st','st.':'st','avenue':'ave','av':'ave','ave.':'ave','road':'rd','rd.':'rd','drive':'dr','dr.':'dr',
          'boulevard':'blvd','blvd.':'blvd','lane':'ln','court':'ct','place':'pl','parkway':'pkwy','highway':'hwy',
          'hiway':'hwy','circle':'cir','terrace':'ter','way':'way','square':'sq','turnpike':'tpke','route':'rt','rte':'rt',
          'north':'n','south':'s','east':'e','west':'w','northeast':'ne','northwest':'nw','southeast':'se','southwest':'sw',
          'suite':'','ste':'','unit':'','building':'','bldg':'','floor':'','fl':''}
NAME_STOP = {'the','of','and','at','a','llc','inc','corp','corporation','co','company','lp','llp','pllc','dba','d','b','a',
             'center','centre','ctr','nursing','rehabilitation','rehab','healthcare','health','care','home','homes',
             'facility','skilled','snf','nh','manor','living','senior','services','service','operating','operations',
             'for','residence','residential','community','transitional','post','acute','subacute','sub','long','term'}

def norm_zip(z):
    z = re.sub(r'\D', '', str(z or ''))
    return z[:5].zfill(5) if z else ''

def norm_addr(a):
    a = str(a or '').lower()
    a = re.sub(r'[#,\.]', ' ', a)
    a = re.sub(r'\b(suite|ste|unit|bldg|building|floor|fl|room|rm)\s*\w+\b', ' ', a)
    toks = [SUFFIX.get(t, t) for t in a.split()]
    return ' '.join(t for t in toks if t)

def house_no(a):
    m = re.match(r'\s*(\d+)', norm_addr(a))
    return m.group(1) if m else ''

def norm_name(n):
    n = str(n or '').lower().replace('&', ' and ')
    n = re.sub(r"[^a-z0-9 ]", ' ', n)
    toks = [t for t in n.split() if t not in NAME_STOP]
    return ' '.join(toks)

def prep(df, id_, name, addr, city, state, zip_):
    out = pd.DataFrame({'id': df[id_].astype(str).str.strip(),
                        'name_raw': df[name], 'addr_raw': df[addr],
                        'city': df[city].astype(str).str.lower().str.strip(),
                        'state': df[state].astype(str).str.upper().str.strip(),
                        'zip5': df[zip_].map(norm_zip)})
    out['name'] = out.name_raw.map(norm_name)
    out['addr'] = out.addr_raw.map(norm_addr)
    out['hno'] = out.addr_raw.map(house_no)
    return out.drop_duplicates('id')

def score_pairs(o, c):
    """o: OSHA establishments, c: CMS facilities (both from prep). Two blocking passes:
    same state + ZIP; and same state + city + house number (catches ZIP typos)."""
    keep = ['id', 'name', 'addr', 'hno', 'city', 'zip5', 'state']
    o, c = o[keep], c[keep]
    b1 = o.merge(c, on=['state', 'zip5'], suffixes=('_o', '_c'))
    b2 = o[o.hno != ''].merge(c[c.hno != ''], on=['state', 'city', 'hno'], suffixes=('_o', '_c'))
    b2 = b2.assign(hno_o=b2.hno, hno_c=b2.hno, city_o=b2.city, city_c=b2.city).drop(columns=['hno', 'city'])
    b1 = b1.assign(zip5_o=b1.zip5, zip5_c=b1.zip5).drop(columns=['zip5'])
    p = pd.concat([b1, b2], ignore_index=True).drop_duplicates(['id_o', 'id_c']).reset_index(drop=True)
    p['name_score'] = [fuzz.token_set_ratio(a, b) / 100 if a and b else 0.0 for a, b in zip(p.name_o, p.name_c)]
    p['addr_score'] = [fuzz.token_sort_ratio(a, b) / 100 if a and b else 0.0 for a, b in zip(p.addr_o, p.addr_c)]
    p['hno_match'] = (p.hno_o == p.hno_c) & (p.hno_o != '')
    p['zip_match'] = p.zip5_o == p.zip5_c
    return p

def tier(r):
    if r.hno_match and r.addr_score >= 0.85 and r.name_score >= 0.60: return 'A'   # same building, name agrees
    if r.hno_match and r.addr_score >= 0.85:                          return 'B'   # same building, name differs (renamed / chain)
    if r.name_score >= 0.90 and r.addr_score >= 0.60:                 return 'B'   # same name, address written differently
    if r.name_score >= 0.95:                                          return 'C'   # same name, same zip, address unclear
    return None

def resolve(p):
    p = p.copy()
    p['tier'] = p.apply(tier, axis=1)
    p = p[p.tier.notna()]
    rank = {'A': 3, 'B': 2, 'C': 1}
    p['total'] = p.tier.map(rank) * 10 + p.name_score + p.addr_score
    p = p.sort_values('total', ascending=False)
    used_o, used_c, keep = set(), set(), []
    for r in p.itertuples():                       # greedy one-to-one
        if r.id_o in used_o or r.id_c in used_c: continue
        used_o.add(r.id_o); used_c.add(r.id_c); keep.append(r.Index)
    return p.loc[keep]


In [15]:
o = prep(osha, 'establishment_id', 'establishment_name', 'street_address', 'city', 'state', 'zip_code')
c = prep(cms, 'ccn', 'provider_name', 'provider_address', 'provider_city', 'provider_state', 'provider_zip')
print(len(o), 'OSHA establishments;', len(c), 'CMS facilities')

8975 OSHA establishments; 14690 CMS facilities


---
## 5. Candidate pairs, scoring, and grading

Two blocking passes limit comparisons to plausible pairs: same state and ZIP; and same state, city, and house number (which catches ZIP typos). Each pair is scored on name similarity (token-set ratio) and address similarity (token-sort ratio), then graded:

| Tier | Rule | Meaning |
|---|---|---|
| **A** | Same house number, address ≥ 0.85, name ≥ 0.60 | Same building, name agrees |
| **B** | Same house number and address ≥ 0.85; *or* name ≥ 0.90 and address ≥ 0.60 | Same building under a different name, or same facility with the address written differently |
| **C** | Name ≥ 0.95 in the same ZIP, address unclear | Probable; use with care |

Each OSHA establishment and each CMS facility is assigned at most once, best match first.

In [16]:
pairs = score_pairs(o, c)
xw = resolve(pairs)
print(f'{len(pairs):,} candidate pairs -> {len(xw):,} matches')
xw.tier.value_counts().sort_index()

20,488 candidate pairs -> 7,757 matches


,count
tier,
A,6406
B,1128
C,223


---
## 6. Match quality

No hand-labeled truth set exists for this linkage. Match quality is checked two ways: whether matched pairs agree on facility size, and by reading a sample.

In [17]:
cov = pd.Series({'OSHA establishments matched': f"{len(xw):,} of {len(o):,} ({len(xw)/len(o):.1%})",
                 'CMS facilities matched': f"{len(xw):,} of {len(c):,} ({len(xw)/len(c):.1%})"})
print(cov.to_string())

OSHA establishments matched     7,757 of 8,975 (86.4%)
CMS facilities matched         7,757 of 14,690 (52.8%)


**Check 1 · Size agreement.** A correct match should link an establishment's employee count to a facility's bed count. The correlation should be positive and similar across tiers; a tier with much weaker correlation contains more wrong matches.

In [18]:
chk = (xw[['id_o', 'id_c', 'tier']]
       .merge(osha[['establishment_id', 'employees']], left_on='id_o', right_on='establishment_id')
       .merge(cms[['ccn', 'certified_beds']], left_on='id_c', right_on='ccn'))
chk = chk[(chk.employees > 0) & (chk.certified_beds > 0)]
size_check = chk.groupby('tier').apply(lambda g: pd.Series({
    'n': len(g), 'spearman_employees_vs_beds': round(stats.spearmanr(g.employees, g.certified_beds).statistic, 3),
    'median_employees_per_bed': round((g.employees / g.certified_beds).median(), 2)}), include_groups=False)
size_check

,n,spearman_employees_vs_beds,median_employees_per_bed
tier,,,
A,6406.0,0.592,1.10
B,1128.0,0.554,1.09
C,223.0,0.424,1.02


**Check 2 · Read a sample.** Thirty random matches, ten per tier. Reading them is the fastest way to find a systematic error.

In [19]:
for t in ['A', 'B', 'C']:
    s = xw[xw.tier == t]
    print(f'===== tier {t} ({len(s):,}) =====')
    for r in s.sample(min(10, len(s)), random_state=0).itertuples():
        print(f'  {r.name_score:.2f} {r.addr_score:.2f} | {r.name_o[:38]:38s} | {r.name_c[:38]:38s}')

---
## 7. Export

The published crosswalk contains identifiers and scores only. Names and addresses remain in the source files, where anyone can join them back.

In [20]:
out = (xw[['id_o', 'id_c', 'tier', 'name_score', 'addr_score', 'hno_match', 'zip_match']]
       .rename(columns={'id_o': 'osha_establishment_id', 'id_c': 'cms_ccn'})
       .sort_values(['tier', 'cms_ccn']))
out['name_score'] = out.name_score.round(3); out['addr_score'] = out.addr_score.round(3)
out.to_csv('osha-cms-crosswalk-nf-2025.csv', index=False)
size_check.to_csv('derived/match-quality-size-check.csv')
pd.Series({'osha_source': SOURCE, 'osha_establishments': len(o), 'cms_facilities': len(c),
           'cms_provider_file_modified': prov_date, 'matches': len(xw),
           **{f'tier_{k}': int(v) for k, v in xw.tier.value_counts().items()}}).rename('value').to_csv('derived/match-summary.csv')
print(len(out), 'rows written')

7757 rows written


---
## 8. Application: does facility injury rate track pressure-injury rate?

The [`nursing-facility-handling`](../nursing-facility-handling/) module reports that facilities with higher pressure-ulcer rates do not have higher injury rates (r ≈ 0.01). This section reproduces that test from the crosswalk. It requires the Form 300A summary file, because rates need hours worked for every facility, including those with no injuries.

DART rate = (cases with days away + cases with restricted or transferred duty) × 200,000 ÷ hours worked.

In [21]:
if SOURCE == '300A':
    mds_url, _ = cms_csv('MDS Quality Measures')
    mds = pd.read_csv(mds_url, dtype=str)
    mc = {x.lower(): x for x in mds.columns}
    g = lambda *n: next(mc[x.lower()] for x in n if x.lower() in mc)
    desc, rtype = g('Measure Description'), g('Resident type')
    pu = mds[mds[desc].str.contains('pressure', case=False, na=False) & mds[rtype].str.contains('long', case=False, na=False)]
    print('measure used:', pu[desc].iloc[0])
    pu = pu.assign(ccn=pu[g('CMS Certification Number (CCN)', 'Federal Provider Number')],
                   pu_rate=pd.to_numeric(pu[g('Four Quarter Average Score')], errors='coerce'))[['ccn', 'pu_rate']]
    a = (out[out.tier.isin(['A', 'B'])]
         .merge(osha, left_on='osha_establishment_id', right_on='establishment_id')
         .merge(pu, left_on='cms_ccn', right_on='ccn'))
    a = a[(a.hours > 10_000) & a.pu_rate.notna()]
    a['dart'] = (a.dafw_cases.fillna(0) + a.djtr_cases.fillna(0)) * 200_000 / a.hours
    res = pd.Series({'n facilities': len(a),
                     'pearson r': round(stats.pearsonr(a.dart, a.pu_rate).statistic, 3),
                     'spearman rho': round(stats.spearmanr(a.dart, a.pu_rate).statistic, 3)})
    res.rename('value').to_csv('derived/dart-vs-pressure-ulcer.csv'); print(res.to_string())
else:
    print('Form 300A summary not available; rates cannot be computed from case detail alone.')

MDS Quality Measures | modified 2026-08-01
measure used: Percentage of long-stay residents with pressure ulcers
n facilities    7370.000
pearson r         -0.001
spearman rho       0.024


---
## Limitations

- **No ground truth.** Match quality is assessed indirectly (size agreement, tier structure, reading a sample). Tier C should be treated as probable, not confirmed.
- **Changes of ownership and name.** Facilities renamed between OSHA submission and the CMS file date appear as tier B or are missed.
- **Multi-building campuses.** One OSHA establishment may correspond to several CMS facilities at nearby addresses, or the reverse. One-to-one assignment keeps only the best pair.
- **Coverage.** OSHA submission requirements depend on establishment size. Unmatched CMS facilities are mostly facilities not required to submit, not failed matches.
- **Identifiers only.** The published table contains no names or addresses. No establishment is characterized in this module.